# 🚀 Rocket Landing AI — Train & Fly in Google Colab

> **👨‍💻 Developed by dug** &nbsp;|&nbsp; ⭐ **GitHub:** [@dug6464](https://github.com/dug6464/rocket-landing-ai)

Train a neural network using **Proximal Policy Optimization (PPO)** and custom **Reward Shaping** to autonomously land a rocket in OpenAI / Farama's Gymnasium `LunarLander-v3` physics environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dug6464/rocket-landing-ai/blob/master/rocket_lander_colab.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-dug6464%2Frocket--landing--ai-blue?logo=github)](https://github.com/dug6464/rocket-landing-ai)

## 1. 📦 Install Dependencies
Install Gymnasium with Box2D physics, Stable-Baselines3, PyTorch, and video rendering utilities.

In [ ]:
# Install dependencies (runs fast in Colab)
!pip install -q swig
!pip install -q gymnasium[box2d] stable-baselines3[extra] imageio[ffmpeg] gradio

## 2. 📥 Clone Repository
Clone the modular project codebase directly from GitHub.

In [ ]:
!git clone https://github.com/dug6464/rocket-landing-ai.git
%cd rocket-landing-ai
!ls -la

## 3. 🧪 Verify Reward Shaping Logic
Run unit tests to verify the shaped reward calculations before training.

In [ ]:
!python -m pytest test_reward.py -v

## 4. 🚀 Train the PPO Rocket Lander
Start training with vectorised environments. In Colab CPU/GPU this takes ~10-15 minutes for 500,000 steps.

In [ ]:
# Train for 500k steps (increase to 1_000_000 for perfect touchdowns)
!python train.py --timesteps 500000 --envs 4

## 5. 🎬 Watch the AI Land & Display Video in Colab
Generate and play an in-notebook video of your trained rocket landing on the pad.

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
import imageio
from IPython.display import HTML, display
from base64 import b64encode
from env import ShapedLunarLander, ENV_ID

# Load trained model
model = PPO.load("models/best_model/best_model.zip")

# Create render environment
env = gym.make(ENV_ID, render_mode="rgb_array")
env = ShapedLunarLander(env)

obs, _ = env.reset(seed=42)
frames = []
done = False
total_reward = 0

while not done:
    frames.append(env.render())
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    done = terminated or truncated

env.close()
print(f"Landing complete! Total Reward: {total_reward:.2f}")

# Save and display video
video_path = "landing.mp4"
imageio.mimsave(video_path, frames, fps=30)

mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f"""
<video width=500 controls autoplay loop>
      <source src="{data_url}" type="video/mp4">
</video>
"""))

## 6. 📊 Run Full Evaluation Suite
Compute landing success rates and plot reward distributions.

In [ ]:
!python evaluate.py --episodes 20

## 7. 🌐 Launch the Interactive Web App
Run Gradio directly inside Colab with a public share link!

In [ ]:
# Launch Gradio web UI with share=True so you get a public URL
from app import demo
demo.launch(share=True)